# 02. Model training and internal validation

This notebook trains the proposed token-wise supervised autoencoder with Transformer-based feature interaction modeling and evaluates the trained models on the held-out ADNI test set.

The notebook performs:

- loading task-specific train/test files generated by `01_preprocessing.ipynb`
- splitting the training data into inner-training and validation subsets
- fitting z-score standardization only on the inner-training subset
- training the model using classification and reconstruction losses
- selecting the best model based on validation AUC
- evaluating the selected model on the held-out test set
- saving internal validation results and ROC curves

## 1. Import libraries

This section imports libraries for data loading, model training, evaluation, and visualization.

In [ ]:
import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
)

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## 2. Reproducibility setting

A fixed random seed is used to reduce stochastic variation across runs.

In [ ]:
GLOBAL_SEED = 17

def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

    try:
        torch.use_deterministic_algorithms(True)
    except Exception as error:
        print("Warning: deterministic algorithms were not fully enabled:", error)

set_seed(GLOBAL_SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 3. File paths and training settings

The processed train/test files are generated by `01_preprocessing.ipynb`.

The held-out test set is used only for final internal validation and is not used for training, early stopping, or model selection.

In [ ]:
PROCESSED_DIR = "../data/processed"
CHECKPOINT_DIR = "../results/checkpoints"
RESULTS_DIR = "../results"
FIGURE_DIR = "../results/figures"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

INNER_VAL_SIZE = 0.10

EPOCHS_FINAL = 200
EARLY_STOP_PATIENCE = 10
EARLY_STOP_MIN_DELTA = 1e-4

TASKS = [
    "AD_vs_CN",
    "AD_vs_MCI",
    "MCI_vs_CN",
    "pMCI_vs_sMCI",
]

## 4. Selected model configuration

The final hyperparameter configuration was selected from the pMCI vs. sMCI hyperparameter search and reused for the remaining binary classification tasks.

In [ ]:
BEST_CFG = {
    "batch_size": 16,
    "d_model": 128,
    "latent_dim": 64,
    "n_heads": 8,
    "n_layers": 2,
    "ff_mult": 2,
    "attn_dropout": 0.25,
    "use_cls": True,

    "mlp_dropout": 0.048,
    "lambda_recon": 0.934,

    "lr": 0.00013,
    "weight_decay": 0.0,

    "enc_hidden": [128, 64, 32],
    "dec_hidden": [32, 64, 128],
    "clf_hidden": [64],
}

## 5. Dataset and DataLoader

The processed feature matrix and binary labels are converted into PyTorch tensors.

In [ ]:
class VectorDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.astype(np.int64), dtype=torch.long)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def make_loader(dataset, batch_size, shuffle, seed=None):
    if seed is not None:
        generator = torch.Generator()
        generator.manual_seed(seed)
    else:
        generator = None

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
        num_workers=0,
        generator=generator,
        persistent_workers=False,
    )

## 6. Model architecture

The model converts scalar clinical variables into feature tokens, compresses them through a token-wise autoencoder bottleneck, and models feature-level interactions using a Transformer encoder.

In [ ]:
def make_mlp_lastdim(in_dim, hidden, out_dim, dropout=0.2):
    layers = []
    prev = in_dim

    for h in hidden:
        layers += [
            nn.Linear(prev, h),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(h),
        ]
        prev = h

    layers.append(nn.Linear(prev, out_dim))
    return nn.Sequential(*layers)


class FeatureTokenizer(nn.Module):
    def __init__(self, n_features: int, d_token: int, token_ln: bool = True):
        super().__init__()

        self.n_features = int(n_features)
        self.d_token = int(d_token)

        self.W = nn.Parameter(torch.randn(self.n_features, self.d_token) * 0.02)
        self.B = nn.Parameter(torch.zeros(self.n_features, self.d_token))

        self.ln = nn.LayerNorm(self.d_token) if token_ln else nn.Identity()

    def forward(self, x):
        x = x.unsqueeze(-1)
        tokens = x * self.W.unsqueeze(0) + self.B.unsqueeze(0)
        return self.ln(tokens)


class SAEFirstAttnClassifier(nn.Module):
    def __init__(
        self,
        n_features: int,
        d_token: int,
        latent_dim: int,
        n_heads: int,
        n_layers: int,
        ff_mult: int,
        attn_dropout: float,
        use_cls: bool,
        sae_enc_hidden=(64, 32),
        sae_dec_hidden=(32, 64),
        clf_hidden=(32, 16),
        mlp_dropout=0.2,
    ):
        super().__init__()

        self.n_features = int(n_features)
        self.d_token = int(d_token)
        self.k = int(latent_dim)
        self.use_cls = bool(use_cls)

        if self.k % int(n_heads) != 0:
            raise ValueError(
                f"latent_dim ({self.k}) must be divisible by n_heads ({n_heads})."
            )

        self.tokenizer = FeatureTokenizer(
            n_features=self.n_features,
            d_token=self.d_token,
            token_ln=True,
        )

        self.sae_encoder = make_mlp_lastdim(
            self.d_token,
            tuple(sae_enc_hidden),
            self.k,
            dropout=mlp_dropout,
        )

        self.sae_decoder = make_mlp_lastdim(
            self.k,
            tuple(sae_dec_hidden),
            self.d_token,
            dropout=mlp_dropout,
        )

        seq_len = self.n_features + (1 if self.use_cls else 0)

        self.pos = nn.Parameter(torch.zeros(1, seq_len, self.k))
        self.cls = nn.Parameter(torch.zeros(1, 1, self.k)) if self.use_cls else None

        ff_dim = int(self.k * ff_mult)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=self.k,
            nhead=int(n_heads),
            dim_feedforward=ff_dim,
            dropout=float(attn_dropout),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            enc_layer,
            num_layers=int(n_layers),
        )

        self.classifier = make_mlp_lastdim(
            self.k,
            tuple(clf_hidden),
            2,
            dropout=mlp_dropout,
        )

        self.out_W = nn.Parameter(torch.randn(self.n_features, self.d_token) * 0.02)
        self.out_b = nn.Parameter(torch.zeros(self.n_features))

        nn.init.normal_(self.pos, mean=0.0, std=0.02)

        if self.use_cls:
            nn.init.normal_(self.cls, mean=0.0, std=0.02)

    def forward(self, x):
        tokens = self.tokenizer(x)
        latent_tokens = self.sae_encoder(tokens)

        if self.use_cls:
            batch_size = x.size(0)
            cls_token = self.cls.expand(batch_size, 1, self.k)
            transformer_input = torch.cat([cls_token, latent_tokens], dim=1)
        else:
            transformer_input = latent_tokens

        contextual_tokens = self.transformer(
            transformer_input + self.pos[:, :transformer_input.size(1), :]
        )

        pooled = contextual_tokens[:, 0, :] if self.use_cls else contextual_tokens.mean(dim=1)
        logits = self.classifier(pooled)

        reconstructed_tokens = self.sae_decoder(latent_tokens)
        x_recon = (
            reconstructed_tokens * self.out_W.unsqueeze(0)
        ).sum(dim=-1) + self.out_b.unsqueeze(0)

        return logits, x_recon

## 7. Loss and evaluation functions

The classification loss is class-weighted cross entropy.  
The auxiliary reconstruction loss is mean squared error.

Validation AUC is used for early stopping, and the held-out test set is evaluated after model selection.

In [ ]:
def get_class_weighted_ce(y_train: np.ndarray):
    c0 = int((y_train == 0).sum())
    c1 = int((y_train == 1).sum())

    w0 = (c0 + c1) / (2.0 * max(c0, 1))
    w1 = (c0 + c1) / (2.0 * max(c1, 1))

    weights = torch.tensor([w0, w1], dtype=torch.float32, device=device)
    return nn.CrossEntropyLoss(weight=weights)


def compute_metrics(y_true, y_prob, y_pred):
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    pre = precision_score(y_true, y_pred, zero_division=0)
    sen = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    spe = tn / (tn + fp + 1e-12)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan

    return {
        "AUC": auc,
        "ACC": acc,
        "BACC": bacc,
        "PRE": pre,
        "SEN": sen,
        "SPE": spe,
        "F1": f1,
    }


@torch.no_grad()
def predict_on_loader(model, loader):
    model.eval()

    y_true = []
    y_prob = []
    y_pred = []

    for xb, yb in loader:
        xb = xb.to(device)

        logits, _ = model(xb)
        prob = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
        pred = (prob >= 0.5).astype(int)

        y_true.append(yb.numpy())
        y_prob.append(prob)
        y_pred.append(pred)

    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)
    y_pred = np.concatenate(y_pred)

    return y_true, y_prob, y_pred

## 8. Training and internal validation function

For each task, the task-specific training split is divided into inner-training and validation subsets.

The scaler is fitted only on the inner-training subset.  
The held-out test set is transformed using the same scaler and evaluated only after the best model is selected.

In [ ]:
def train_and_evaluate_task(task_name, cfg, seed, feature_cols=None, save_checkpoint=True):
    set_seed(seed)

    train_path = f"{PROCESSED_DIR}/{task_name}_train.csv"
    test_path = f"{PROCESSED_DIR}/{task_name}_test.csv"

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    if feature_cols is None:
        feature_cols = train_df.drop(columns=["label"]).columns.tolist()

    y_train_all = train_df["label"].values.astype(np.int64)
    X_train_all = train_df[feature_cols].values.astype(np.float32)

    y_test = test_df["label"].values.astype(np.int64)
    X_test = test_df[feature_cols].values.astype(np.float32)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_all,
        y_train_all,
        test_size=INNER_VAL_SIZE,
        stratify=y_train_all,
        random_state=seed,
    )

    scaler = StandardScaler()

    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    batch_size = int(cfg["batch_size"])

    train_loader = make_loader(
        VectorDataset(X_tr_scaled, y_tr),
        batch_size=batch_size,
        shuffle=True,
        seed=seed,
    )

    val_loader = make_loader(
        VectorDataset(X_val_scaled, y_val),
        batch_size=batch_size,
        shuffle=False,
        seed=seed,
    )

    test_loader = make_loader(
        VectorDataset(X_test_scaled, y_test),
        batch_size=batch_size,
        shuffle=False,
        seed=seed,
    )

    model = SAEFirstAttnClassifier(
        n_features=X_tr_scaled.shape[1],
        d_token=int(cfg["d_model"]),
        latent_dim=int(cfg["latent_dim"]),
        n_heads=int(cfg["n_heads"]),
        n_layers=int(cfg["n_layers"]),
        ff_mult=int(cfg["ff_mult"]),
        attn_dropout=float(cfg["attn_dropout"]),
        use_cls=bool(cfg["use_cls"]),
        sae_enc_hidden=tuple(cfg["enc_hidden"]),
        sae_dec_hidden=tuple(cfg["dec_hidden"]),
        clf_hidden=tuple(cfg["clf_hidden"]),
        mlp_dropout=float(cfg["mlp_dropout"]),
    ).to(device)

    ce_loss = get_class_weighted_ce(y_tr)
    mse_loss = nn.MSELoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(cfg["lr"]),
        weight_decay=float(cfg["weight_decay"]),
    )

    lambda_recon = float(cfg["lambda_recon"])

    best_auc = -np.inf
    best_state = None
    best_epoch = None
    patience_count = 0

    history = []

    for epoch in range(1, EPOCHS_FINAL + 1):
        model.train()

        epoch_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits, recon = model(xb)

            loss_cls = ce_loss(logits, yb)
            loss_rec = mse_loss(recon, xb)
            loss = loss_cls + lambda_recon * loss_rec

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        y_val_true, y_val_prob, _ = predict_on_loader(model, val_loader)

        try:
            val_auc = roc_auc_score(y_val_true, y_val_prob)
        except Exception:
            val_auc = np.nan

        history.append({
            "task": task_name,
            "epoch": epoch,
            "train_loss": float(np.mean(epoch_losses)),
            "val_auc": val_auc,
        })

        improved = (not np.isnan(val_auc)) and (val_auc > best_auc + EARLY_STOP_MIN_DELTA)

        if improved:
            best_auc = float(val_auc)
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            patience_count = 0
        else:
            patience_count += 1

            if patience_count >= EARLY_STOP_PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    y_true, y_prob, y_pred = predict_on_loader(model, test_loader)
    test_metrics = compute_metrics(y_true, y_prob, y_pred)

    checkpoint = {
        "task": task_name,
        "model_state_dict": model.state_dict(),
        "config": cfg,
        "n_features": X_tr_scaled.shape[1],
        "feature_cols": feature_cols,
        "scaler": scaler,
        "best_val_auc": best_auc,
        "best_epoch": best_epoch,
        "seed": seed,
    }

    checkpoint_path = None

    if save_checkpoint:
        checkpoint_path = f"{CHECKPOINT_DIR}/{task_name}_best.pt"
        torch.save(checkpoint, checkpoint_path)

    result = {
        "task": task_name,
        "best_val_auc": best_auc,
        "best_epoch": best_epoch,
        **test_metrics,
        "checkpoint_path": checkpoint_path,
    }

    history_df = pd.DataFrame(history)

    roc_info = {
        "task": task_name,
        "y_true": y_true,
        "y_prob": y_prob,
    }

    return result, history_df, roc_info

## 9. Train and evaluate all tasks

The same selected model configuration is applied to all four binary classification tasks.

In [ ]:
all_results = []
all_histories = []
roc_outputs = []

for task_name in TASKS:
    print(f"\nRunning task: {task_name}")

    result, history_df, roc_info = train_and_evaluate_task(
        task_name=task_name,
        cfg=BEST_CFG,
        seed=GLOBAL_SEED,
    )

    all_results.append(result)
    all_histories.append(history_df)
    roc_outputs.append(roc_info)

    print(
        f"{task_name} | best epoch: {result['best_epoch']} | "
        f"best validation AUC: {result['best_val_auc']:.4f} | "
        f"test AUC: {result['AUC']:.4f} | test ACC: {result['ACC']:.4f}"
    )

internal_results_df = pd.DataFrame(all_results)
training_history_df = pd.concat(all_histories, ignore_index=True)

internal_results_df

## 10. Save internal validation results

Internal test metrics and epoch-wise validation history are saved locally.

In [ ]:
internal_results_path = f"{RESULTS_DIR}/internal_test_results.csv"
history_path = f"{RESULTS_DIR}/training_history.csv"

internal_results_df.to_csv(internal_results_path, index=False)
training_history_df.to_csv(history_path, index=False)

print(f"Saved internal validation results to: {internal_results_path}")
print(f"Saved training history to: {history_path}")

## 11. Plot ROC curves

ROC curves are generated for the four internal ADNI test tasks.

In [ ]:
plt.figure(figsize=(7, 6))

for roc_info in roc_outputs:
    task_name = roc_info["task"]
    y_true = roc_info["y_true"]
    y_prob = roc_info["y_prob"]

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)

    label = task_name.replace("_", " ")

    plt.plot(
        fpr,
        tpr,
        label=f"{label} (AUC = {auc:.4f})",
    )

plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Internal ADNI ROC curves")
plt.legend(loc="lower right")
plt.tight_layout()

roc_path = f"{FIGURE_DIR}/internal_roc_curves.png"
plt.savefig(roc_path, dpi=300)
plt.show()

print(f"Saved ROC curve figure to: {roc_path}")

## 12. Expected outputs

After running this notebook, the following files should be generated locally:

```text
results/internal_test_results.csv
results/training_history.csv
results/figures/internal_roc_curves.png

results/checkpoints/AD_vs_CN_best.pt
results/checkpoints/AD_vs_MCI_best.pt
results/checkpoints/MCI_vs_CN_best.pt
results/checkpoints/pMCI_vs_sMCI_best.pt